# 09 — Frozen 2013 official-test evaluation

Evaluate baseline v2 once using its frozen checkpoint, normalization, and validation-selected threshold.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.6"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
TRAINING_SEED = 20260913

BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 8
LEARNING_RATE = 1e-3
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
MINIMUM_LEARNING_RATE = 1e-5

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
DRIVE_ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "2013_baseline_v1"
    / "normalization.json"
)
EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "2013_baseline_v2"
)
CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "best_model.pt"
)
METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_metrics.json"
)
HISTORY_PATH = (
    EXPERIMENT_DIRECTORY
    / "training_history.csv"
)
TEST_METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "official_test_metrics.json"
)
FROZEN_THRESHOLD = 0.523741602897644
FROZEN_BEST_EPOCH = 45

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_2013_baseline"
)
LOCAL_CHECKPOINT_PATH = Path(
    "/content/best_model.pt"
)

for required_path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    DRIVE_ARCHIVE_PATH,
    NORMALIZATION_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required path: "
            f"{required_path}"
        )

if TEST_METRICS_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite {TEST_METRICS_PATH}"
    )

print("package:", PACKAGE_PATH)
print("normalization:", NORMALIZATION_PATH)
print("experiment:", EXPERIMENT_DIRECTORY)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl
normalization: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1/normalization.json
experiment: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v2


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl'], returncode=0)

In [4]:
import json
import random

import numpy as np
import pandas as pd
import torch

import tornado_detection

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select a Colab GPU "
        "runtime, restart, and run all cells."
    )

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

expected_normalization = {
    "package_version": PACKAGE_VERSION,
    "year": 2013,
    "official_source_split": "train",
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": VALIDATION_SEED,
    "variables": ["DBZ", "VEL"],
    "channel_order": [
        "DBZ_sweep_0",
        "DBZ_sweep_1",
        "VEL_sweep_0",
        "VEL_sweep_1",
    ],
    "tensor_shape": [120, 240, 4],
    "training_frame_count": 11056,
    "training_file_count": 2764,
    "training_positive_frame_count": 445,
    "label_mismatch_count": 0,
}

mismatches = {
    key: {
        "expected": value,
        "actual": normalization.get(key),
    }
    for key, value
    in expected_normalization.items()
    if normalization.get(key) != value
}

if mismatches:
    raise AssertionError(
        "Normalization provenance mismatch: "
        f"{mismatches}"
    )

channel_means = np.asarray(
    normalization["means"],
    dtype=np.float32,
)
channel_stds = np.asarray(
    normalization[
        "standard_deviations"
    ],
    dtype=np.float32,
)

assert channel_means.shape == (4,)
assert channel_stds.shape == (4,)
assert np.isfinite(channel_means).all()
assert np.isfinite(channel_stds).all()
assert np.all(channel_stds > 0)

print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print("torch:", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print(
    "means:",
    channel_means.tolist(),
)
print(
    "stds:",
    channel_stds.tolist(),
)


tornado_detection: 0.1.6
torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB
means: [23.446088790893555, 23.32160186767578, -2.4620916843414307, -2.225398540496826]
stds: [14.11445426940918, 14.181056022644043, 16.997556686401367, 17.79329490661621]


In [5]:
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
)

canonical_index = (
    load_canonical_frame_index(
        MANIFESTS_ROOT
    )
)
assigned_index = assign_model_splits(
    canonical_index,
    validation_fraction=(
        VALIDATION_FRACTION
    ),
    seed=VALIDATION_SEED,
)

year_index = assigned_index.loc[
    assigned_index["year"].eq(2013)
].copy()

train_index = (
    year_index.loc[
        year_index["model_split"].eq(
            "train"
        )
    ]
    .sort_values("frame_id")
    .reset_index(drop=True)
)
validation_index = (
    year_index.loc[
        year_index["model_split"].eq(
            "validation"
        )
    ]
    .sort_values("frame_id")
    .reset_index(drop=True)
)
official_test_index = (
    year_index.loc[
        year_index["model_split"].eq(
            "test"
        )
    ]
    .sort_values("frame_id")
    .reset_index(drop=True)
)

assert len(train_index) == 11_056
assert len(validation_index) == 2_936
assert len(official_test_index) == 2_292
assert int(
    train_index["frame_label"].sum()
) == 445
assert int(
    validation_index[
        "frame_label"
    ].sum()
) == 143
assert int(
    official_test_index[
        "frame_label"
    ].sum()
) == 157

internal_groups = pd.concat(
    [
        train_index[
            [
                "validation_group_key",
                "model_split",
            ]
        ],
        validation_index[
            [
                "validation_group_key",
                "model_split",
            ]
        ],
    ],
    ignore_index=True,
).drop_duplicates()

crossing_groups = (
    internal_groups.groupby(
        "validation_group_key"
    )["model_split"]
    .nunique()
)

assert int(
    (crossing_groups > 1).sum()
) == 0

print(
    "train frames:",
    f"{len(train_index):,}",
)
print(
    "train positives:",
    int(
        train_index[
            "frame_label"
        ].sum()
    ),
)
print(
    "validation frames:",
    f"{len(validation_index):,}",
)
print(
    "validation positives:",
    int(
        validation_index[
            "frame_label"
        ].sum()
    ),
)
print(
    "official test held out:",
    f"{len(official_test_index):,}",
)
print("crossing groups: 0")


train frames: 11,056
train positives: 445
validation frames: 2,936
validation positives: 143
official test held out: 2,292
crossing groups: 0


In [6]:
import shutil
import tarfile
import time

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if LOCAL_CHECKPOINT_PATH.exists():
    LOCAL_CHECKPOINT_PATH.unlink()

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    DRIVE_ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter()
    - copy_started
)

required_members = set(
    official_test_index[
        "archive_member"
    ].unique()
)

extracted_members = set()
extraction_started = (
    time.perf_counter()
)

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in required_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT
            / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                f"Could not extract "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(
            member.name
        )

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing_members = (
    required_members
    - extracted_members
)

if missing_members:
    raise RuntimeError(
        "Missing internal train/validation "
        f"members: "
        f"{sorted(missing_members)[:10]}"
    )

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "extracted files:",
    f"{len(extracted_members):,}",
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)


copy seconds: 6.175
extracted files: 573
extraction seconds: 9.416


In [7]:
import xarray as xr
from torch.utils.data import DataLoader, Dataset

from tornado_detection.data import (
    build_frame_tensor,
)


class RadarFrameDataset(Dataset):
    def __init__(
        self,
        rows,
        root,
        means,
        stds,
    ):
        self.rows = (
            rows[
                [
                    "frame_id",
                    "archive_member",
                    "frame_index",
                    "frame_label",
                ]
            ]
            .reset_index(drop=True)
            .copy()
        )
        self.root = root
        self.means = means.reshape(
            1,
            1,
            4,
        )
        self.stds = stds.reshape(
            1,
            1,
            4,
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        path = (
            self.root
            / str(row["archive_member"])
        )

        with xr.open_dataset(
            path,
            engine="netcdf4",
        ) as dataset:
            result = build_frame_tensor(
                dataset,
                int(row["frame_index"]),
            )

        expected_label = int(
            row["frame_label"]
        )

        if result.label != expected_label:
            raise AssertionError(
                "Label mismatch for "
                f"{row['frame_id']}"
            )

        values = (
            result.values
            - self.means
        ) / self.stds

        values = np.nan_to_num(
            values,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(
            np.float32,
            copy=False,
        )

        tensor = torch.from_numpy(
            values
        ).permute(
            2,
            0,
            1,
        ).contiguous()

        label = torch.tensor(
            [expected_label],
            dtype=torch.float32,
        )

        return tensor, label


test_dataset = RadarFrameDataset(
    official_test_index,
    EXTRACTION_ROOT,
    channel_means,
    channel_stds,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

smoke_inputs, smoke_labels = next(
    iter(test_loader)
)

assert smoke_inputs.shape[1:] == (
    4,
    120,
    240,
)
assert torch.isfinite(
    smoke_inputs
).all()

print(
    "official-test batches:",
    len(test_loader),
)
print(
    "smoke batch:",
    tuple(smoke_inputs.shape),
)


official-test batches: 36
smoke batch: (64, 4, 120, 240)


In [8]:
import torch.nn as nn


class RadarBaselineCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                4,
                16,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        inputs: torch.Tensor,
    ) -> torch.Tensor:
        return self.classifier(
            self.features(inputs)
        )


model = RadarBaselineCNN().to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "parameters:",
    f"{parameter_count:,}",
)


parameters: 106,385


In [9]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

assert int(
    checkpoint["epoch"]
) == FROZEN_BEST_EPOCH

model.load_state_dict(
    checkpoint["model_state_dict"]
)
model.eval()

labels = []
probabilities = []

evaluation_started = time.perf_counter()

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(
            device,
            non_blocking=True,
        )

        logits = model(inputs)

        labels.extend(
            targets.numpy()
            .reshape(-1)
            .tolist()
        )
        probabilities.extend(
            logits.sigmoid()
            .cpu()
            .numpy()
            .reshape(-1)
            .tolist()
        )

evaluation_seconds = (
    time.perf_counter()
    - evaluation_started
)

labels = np.asarray(
    labels,
    dtype=np.int64,
)
probabilities = np.asarray(
    probabilities,
    dtype=np.float64,
)

assert len(labels) == 2_292
assert int(labels.sum()) == 157

predictions = (
    probabilities
    >= FROZEN_THRESHOLD
).astype(np.int64)

pr_auc = float(
    average_precision_score(
        labels,
        probabilities,
    )
)
roc_auc = float(
    roc_auc_score(
        labels,
        probabilities,
    )
)
precision = float(
    precision_score(
        labels,
        predictions,
        zero_division=0,
    )
)
recall = float(
    recall_score(
        labels,
        predictions,
        zero_division=0,
    )
)
f1 = float(
    2
    * precision
    * recall
    / max(
        precision + recall,
        1e-12,
    )
)

matrix = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
)
true_negative, false_positive, false_negative, true_positive = (
    int(value)
    for value in matrix.ravel()
)

print(
    "official-test PR-AUC:",
    pr_auc,
)
print(
    "official-test ROC-AUC:",
    roc_auc,
)
print(
    "frozen threshold:",
    FROZEN_THRESHOLD,
)
print("precision:", precision)
print("recall:", recall)
print("F1:", f1)
print(
    "confusion matrix:",
    matrix.tolist(),
)
print(
    "evaluation seconds:",
    round(evaluation_seconds, 3),
)


official-test PR-AUC: 0.2470871900551703
official-test ROC-AUC: 0.8556795298259222
frozen threshold: 0.523741602897644
precision: 0.3157894736842105
recall: 0.535031847133758
F1: 0.39716312056737585
confusion matrix: [[1953, 182], [73, 84]]
evaluation seconds: 13.662


In [10]:
import datetime

metrics = {
    "artifact_kind": (
        "2013_official_test_metrics"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": PACKAGE_VERSION,
    "selected_model": (
        "2013_baseline_v2"
    ),
    "checkpoint_epoch": (
        FROZEN_BEST_EPOCH
    ),
    "threshold_source": (
        "2013_baseline_v2 validation"
    ),
    "frozen_threshold": (
        FROZEN_THRESHOLD
    ),
    "official_test_evaluated": True,
    "test_frame_count": int(
        len(labels)
    ),
    "test_positive_count": int(
        labels.sum()
    ),
    "test_pr_auc": pr_auc,
    "test_roc_auc": roc_auc,
    "test_precision": precision,
    "test_recall": recall,
    "test_f1": f1,
    "true_negative": true_negative,
    "false_positive": false_positive,
    "false_negative": false_negative,
    "true_positive": true_positive,
    "evaluation_seconds": float(
        evaluation_seconds
    ),
}

TEST_METRICS_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
)
print(
    "wrote:",
    TEST_METRICS_PATH,
)


{
  "artifact_kind": "2013_official_test_metrics",
  "checkpoint_epoch": 45,
  "created_at_utc": "2026-09-13T22:08:27.200835+00:00",
  "evaluation_seconds": 13.661631458000556,
  "false_negative": 73,
  "false_positive": 182,
  "frozen_threshold": 0.523741602897644,
  "official_test_evaluated": true,
  "package_version": "0.1.6",
  "selected_model": "2013_baseline_v2",
  "test_f1": 0.39716312056737585,
  "test_frame_count": 2292,
  "test_positive_count": 157,
  "test_pr_auc": 0.2470871900551703,
  "test_precision": 0.3157894736842105,
  "test_recall": 0.535031847133758,
  "test_roc_auc": 0.8556795298259222,
  "threshold_source": "2013_baseline_v2 validation",
  "true_negative": 1953,
  "true_positive": 84
}
wrote: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v2/official_test_metrics.json


In [11]:
shutil.rmtree(EXTRACTION_ROOT)
LOCAL_ARCHIVE_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()
assert TEST_METRICS_PATH.is_file()

print(
    "Removed all Colab-local test artifacts"
)
print(
    "Preserved:",
    TEST_METRICS_PATH,
)


Removed all Colab-local test artifacts
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v2/official_test_metrics.json
